# lean v4 — self-distillation generation pass

v1/v2/v3 all trained on GSM8K's own gold-label style (uniformly terse,
~52 words regardless of problem difficulty) and all landed around
45-51% accuracy vs. the base model's 69%. The likely cause: training on a
fixed-length terse target teaches the model to always answer briefly, which
hurts on harder problems that need more reasoning.

v4 tries a different recipe: generate the **base model's own reasoning**
on all 7,473 training questions, keep only the ones where it already gets
the right answer (self-distillation on verified-correct output only, never
on its own mistakes), then trim filler from those — not compress to a fixed
template. Token count should then scale with problem difficulty instead of
collapsing everything to the same length.

This notebook does step 1: batched generation on all 7,473 questions.
Batching matters here — unbatched sequential generation at ~3-5s/example
would take 6+ hours; batched generation should be well under an hour on a
free T4.

1. New Notebook → Settings → Accelerator: **GPU T4 x2** (or P100), Internet on.
2. Add Data → reuse the `lean-pairs` dataset (has `raw.jsonl`), or upload `data/raw.jsonl` fresh.
3. Run all cells. Output: `distill_raw.jsonl` with {question, gold, generation} for all 7,473 rows.

In [ ]:
!pip install -q transformers accelerate

In [ ]:
CFG = {
    "base_model": "Qwen/Qwen2.5-1.5B-Instruct",
    "raw_path": "/kaggle/input/datasets/johnandreimartinez/lean-pairs/raw.jsonl",  # adjust to your uploaded dataset
    "out_path": "/kaggle/working/distill_raw.jsonl",
    "batch_size": 32,
    "max_new_tokens": 400,
}

In [ ]:
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(CFG["base_model"])
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    CFG["base_model"], torch_dtype=torch.float16, device_map="cuda"
)
model.eval()

In [ ]:
rows = [json.loads(l) for l in open(CFG["raw_path"], encoding="utf-8")]
print("total questions:", len(rows))

In [ ]:
import re

def gold_answer(answer_field):
    return answer_field.split("####")[-1].strip().replace(",", "")

out_f = open(CFG["out_path"], "w", encoding="utf-8")
bs = CFG["batch_size"]

with torch.no_grad():
    for start in range(0, len(rows), bs):
        batch = rows[start:start + bs]
        prompts = [r["question"] for r in batch]
        inputs = tok(prompts, return_tensors="pt", padding=True).to("cuda")
        out = model.generate(
            **inputs,
            max_new_tokens=CFG["max_new_tokens"],
            do_sample=False,
            pad_token_id=tok.pad_token_id,
        )
        gen_only = out[:, inputs["input_ids"].shape[1]:]
        texts = tok.batch_decode(gen_only, skip_special_tokens=True)
        for r, text in zip(batch, texts):
            out_f.write(json.dumps({
                "question": r["question"],
                "gold": gold_answer(r["answer"]),
                "generation": text,
            }) + "\n")
        out_f.flush()
        if (start // bs) % 10 == 0:
            print(f"done {start + len(batch)}/{len(rows)}")

out_f.close()
print("wrote", CFG["out_path"])

In [ ]:
# Download /kaggle/working/distill_raw.jsonl, then run data/build_v4.py locally on it.